# RNA-seq alignment
This notebook will create all the necessary files, scripts and folders to pre-process the aforementioned project. Is designed to be deployed in a system running SLURM. The majority of the scripts and heavy-lifting processes are wrapped up in sbatch scripts. As an end user, in order to pre-process your samples provided in the spreadsheet, you will simply need to run the notebook.
#### Create necessary folder(s)

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/path/to/project")
SOFTWARE_DIR = Path("/path/to/software")
UCSC_BIN_DIR = Path("/path/to/ucsc/bin")
CONDA_ACTIVATE = "/path/to/miniconda2/bin/activate"
CWL_ENV = "cwl10"
QC_ENV = "alex"
SLURM_PARTITION = "all"
DOWNLOAD_HOST = "user@transfer-host.example.edu"
SLURM_MAIL_USER = "user@example.edu"

os.environ.update({
    "PROJECT_DIR": str(PROJECT_DIR),
    "SOFTWARE_DIR": str(SOFTWARE_DIR),
    "UCSC_BIN_DIR": str(UCSC_BIN_DIR),
    "CONDA_ACTIVATE": CONDA_ACTIVATE,
    "CWL_ENV": CWL_ENV,
    "QC_ENV": QC_ENV,
    "SLURM_PARTITION": SLURM_PARTITION,
    "DOWNLOAD_HOST": DOWNLOAD_HOST,
    "SLURM_MAIL_USER": SLURM_MAIL_USER,
})


In [ ]:
%%bash
mkdir -p "$PROJECT_DIR/data/rna_seq/metadata"
mkdir -p "$PROJECT_DIR/data/rna_seq/raw_reads"
mkdir -p "$PROJECT_DIR/data/rna_seq/processed_raw_reads"
mkdir -p "$PROJECT_DIR/processing/rna_seq/scripts"
mkdir -p "$PROJECT_DIR/processing/rna_seq/jsons"
mkdir -p "$PROJECT_DIR/processing/rna_seq/logs"

Save metadata file

In [ ]:
%%writefile $PROJECT_DIR/data/rna_seq/metadata/rna_seq_download_metadata.RNA-seq.txt
Sequencing core project	Sequencing core library name	Name	Paired-end or single-end	Genome	Library type	Strand specificity
RNA-seq	NT-16-1	RNA-seq_NT-1	PE	hg38	RNA-seq	unstranded
RNA-seq	NT-16-2	RNA-seq_NT-2	PE	hg38	RNA-seq	unstranded
RNA-seq	NT-16-3	RNA-seq_NT-3	PE	hg38	RNA-seq	unstranded


#### Download FASTQ from other
Create file to download FASTQ files

In [ ]:
%%writefile $PROJECT_DIR/processing/rna_seq/scripts/download_RNA-seq.sh
#!/bin/bash
METADATA=${PROJECT_DIR}/data/rna_seq/metadata/rna_seq_download_metadata.RNA-seq.txt
DATA_HOME=${PROJECT_DIR}/data/rna_seq
mkdir -p ${DATA_HOME}/raw_reads/


Execute file to download files

In [ ]:
%%script --out blocking_job_str bash
sbatch -o "$PROJECT_DIR/processing/rna_seq/logs/RNA-seq_download_fastq_files.out" \
 -p "$SLURM_PARTITION" \
 --wrap="ssh $DOWNLOAD_HOST 'sh $PROJECT_DIR/processing/rna_seq/scripts/download_RNA-seq.sh' "

Extract blocking job id

In [ ]:
import re
blocking_job = re.match('Submitted batch job (\d+).*', blocking_job_str).group(1)

#### Merge lanes of FASTQ files

In [ ]:
%%writefile $PROJECT_DIR/processing/rna_seq/scripts/merge_lanes_RNA-seq.sh
#!/bin/bash
#SBATCH --array=0-3%20
ORDER=RNA-seq
RAW_DATA_DIR=${PROJECT_DIR}/data/rna_seq/raw_reads/${ORDER}
PROCESSED_DATA_DIR=${PROJECT_DIR}/data/rna_seq/processed_raw_reads/${ORDER}
METADATA=${PROJECT_DIR}/data/rna_seq/metadata/rna_seq_download_metadata.RNA-seq.txt

mkdir -p ${PROCESSED_DATA_DIR}
cd ${PROCESSED_DATA_DIR}

seq_name_header=$(/bin/grep -Eoi "sequencing.?core.?library.?name" ${METADATA})
if [[ $? == 1 ]];
then
    echo -e "ERROR: Sequencing core library name not found in ${METADATA}"
    exit 1
fi

name_header=$(/bin/grep -Poi "\tname\t" ${METADATA})
if [[ $? == 1 ]];
then
    echo -e "ERROR: Library Name column not found in ${METADATA}"
    exit 1
fi
name_header=$(echo ${name_header} | cut -f2)

seq_type_header=$(head -1 ${METADATA} | /bin/grep -Poi "paired.?end.?or.?single.?end")
if [[ $? == 1 ]];
then
    echo -e "ERROR: Paired-end or single-end column not found in ${METADATA}"
    exit 1
fi

sample_seq_name=$(${SOFTWARE_DIR}/bin/print_tab_cols.awk -v cols="${seq_name_header}" ${METADATA} \
    | awk -v SLURM_ARRAY_TASK_ID=${SLURM_ARRAY_TASK_ID} 'NR==SLURM_ARRAY_TASK_ID+1{print}');
sample_name=$(${SOFTWARE_DIR}/bin/print_tab_cols.awk -v cols="${name_header}" ${METADATA} \
    | awk -v SLURM_ARRAY_TASK_ID=${SLURM_ARRAY_TASK_ID} 'NR==SLURM_ARRAY_TASK_ID+1{print}');
seq_type=$(${SOFTWARE_DIR}/bin/print_tab_cols.awk -v cols="${seq_type_header}" ${METADATA} \
    | awk -v SLURM_ARRAY_TASK_ID=${SLURM_ARRAY_TASK_ID} 'NR==SLURM_ARRAY_TASK_ID+1{print}');

for read_pair in R1 R2;
do
    sample_files=$(/bin/ls ${RAW_DATA_DIR}/${sample_seq_name/ /}_${read_pair}_[0-9][0-9][0-9]* 2> /dev/null)
    if [[ $? != 0 ]]; # If no samples found with that read_pair, continue
    then
        continue;
    fi
    if [[ ${read_pair} == "R1" || (${seq_type/ /} == "PE" || ${seq_type/ /} == "pe") ]];
    then
        # Merge all lanes
        merged=$(basename $(echo ${sample_files} | awk '{print $1}') | sed -e 's/_[0-9]\{3\}_/_/')
        cat ${sample_files} > ${merged};

        # Rename samples with our sample Names
        dest_filename=$(basename $(echo ${merged} | awk '{print $1}') | sed -r 's/\_S[0-9]+//; s/\_(R1|R2|UMI)\_/\.\1\./; s/\.[0-9]+\.fastq/\.fastq/')
        mv ${merged} ${dest_filename}

        cleaned_dest_filename=${dest_filename/${sample_seq_name/ /}/${sample_name/ /}}

        if [[ ${seq_type/ /} == "SE" || ${seq_type/ /} == "se" ]];
        then
            cleaned_dest_filename=${cleaned_dest_filename/.R1/}
        fi
        
        mv ${dest_filename} ${cleaned_dest_filename}
    fi
done


Execute file to merge lanes of FASTQ files

In [ ]:
%%script --out blocking_job_str bash -s "$blocking_job"
sbatch -o "$PROJECT_DIR/processing/rna_seq/logs/RNA-seq_merge_fastq_files_%a.out" \
 -p "$SLURM_PARTITION" \
 --array 0-2%20 \
 "$PROJECT_DIR/processing/rna_seq/scripts/merge_lanes_RNA-seq.sh"

Extract blocking job id

In [ ]:
import re
blocking_job = re.match('Submitted batch job (\d+).*', blocking_job_str).group(1)

#### Create JSON files for CWL pipeline files

In [ ]:
%%writefile $PROJECT_DIR/processing/rna_seq/scripts/cwl_json_gen_RNA-seq.sh
#!/bin/bash
ORDER=RNA-seq
PROCESSED_DATA_DIR=${PROJECT_DIR}/data/rna_seq/processed_raw_reads/${ORDER}
METADATA=${PROJECT_DIR}/data/rna_seq/metadata/rna_seq_download_metadata.RNA-seq.txt

python ${SOFTWARE_DIR}/cwl/GGR-cwl/v1.0/json-generator/run.py \
    -m ${METADATA} \
    -d ${PROCESSED_DATA_DIR} \
    -o ${PROJECT_DIR}/processing/rna_seq/jsons \
    -t rna-seq \
    --fastq-gzipped \
    --mem 48000 \
    --nthreads 24 \
    --separate-jsons \
    --skip-star-2pass \


Execute file to create JSON files

In [ ]:
%%script --out blocking_job_str bash -s "$blocking_job"
source "$CONDA_ACTIVATE" "$CWL_ENV"
sbatch -o "$PROJECT_DIR/processing/rna_seq/logs/RNA-seq_cwl_json_gen.out" \
 -p "$SLURM_PARTITION" \
 --depend afterok:$1 \
 "$PROJECT_DIR/processing/rna_seq/scripts/cwl_json_gen_RNA-seq.sh"

Extract blocking job id

In [ ]:
import re
blocking_job = re.match('Submitted batch job (\d+).*', blocking_job_str).group(1)

#### Create SLURM array master bash file for pe-unstranded-with-sjdb samples

In [ ]:
%%writefile $PROJECT_DIR/processing/rna_seq/scripts/RNA-seq-pe-unstranded-with-sjdb.sh
#!/bin/bash
#SBATCH --job-name=cwl_rna_seq
#SBATCH --output=${PROJECT_DIR}/processing/rna_seq/logs/RNA-seq-pe-unstranded-with-sjdb-%a.out
#SBATCH --mail-user=${SLURM_MAIL_USER}
#SBATCH --mail-type=FAIL,END
#SBATCH --mem=48000
#SBATCH --cpus-per-task=24


export PATH="${SOFTWARE_DIR}/bin:$PATH"
export PATH="${UCSC_BIN_DIR}:$PATH"
export PATH="${SOFTWARE_DIR}/cwl/bin:$PATH"
export PATH="${SOFTWARE_DIR}/preseq_v2.0:$PATH"
export PATH="${SOFTWARE_DIR}/rsem-1.2.21/:$PATH"
export PATH="${SOFTWARE_DIR}/STAR-STAR_2.4.1a/bin/Linux_x86_64/:$PATH"
export PATH="${SOFTWARE_DIR}/subread-1.4.6-p4-Linux-x86_64/bin/:$PATH"
export PATH="${SOFTWARE_DIR}/bamtools-2.2.3/bin/:$PATH"

export PATH="${SOFTWARE_DIR}/miniconda2/envs/${CWL_ENV}/bin:$PATH"

module load bedtools2
module load fastqc
module load samtools
module load bowtie2
module load java

# For Fastqc
export DISPLAY=:0.0

# Make sure temporary files and folders are created in a specific folder
mkdir -p ${PROJECT_DIR}/processing/rna_seq/tmpdirs/tmp-RNA-seq-pe-unstranded-with-sjdb-${SLURM_ARRAY_TASK_ID}-
export TMPDIR="${PROJECT_DIR}/processing/rna_seq/tmpdirs/tmp-RNA-seq-pe-unstranded-with-sjdb-${SLURM_ARRAY_TASK_ID}-"

cwltool --debug \
    --non-strict \
    --preserve-environment PATH \
    --preserve-environment DISPLAY \
    --preserve-environment TMPDIR \
    --outdir ${PROJECT_DIR}/processing/rna_seq/RNA-seq-pe-unstranded-with-sjdb  \
    --no-container \
    ${SOFTWARE_DIR}/cwl/GGR-cwl/v1.0/RNA-seq_pipeline/pipeline-pe-unstranded-with-sjdb.cwl \
    ${PROJECT_DIR}/processing/rna_seq/jsons/rna_seq_download_metadata.RNA-seq-pe-unstranded-with-sjdb-${SLURM_ARRAY_TASK_ID}.json


# Delete any tmpdir not removed by cwltool
rm -rf ${PROJECT_DIR}/processing/rna_seq/tmpdirs/tmp-RNA-seq-pe-unstranded-with-sjdb-${SLURM_ARRAY_TASK_ID}-


Execute SLURM array master file

In [ ]:
%%script --out blocking_job_str bash -s "$blocking_job"
source "$CONDA_ACTIVATE" "$CWL_ENV"
sbatch -p "$SLURM_PARTITION" \
 --depend afterok:$1 \
 --array 0-2%20 \
 "$PROJECT_DIR/processing/rna_seq/scripts/RNA-seq-pe-unstranded-with-sjdb.sh"

Extract blocking job id

In [ ]:
import re
blocking_job = re.match('Submitted batch job (\d+).*', blocking_job_str).group(1)

#### Create QC generating script

In [ ]:
%%writefile $PROJECT_DIR/processing/rna_seq/scripts/generate_qc_cell_RNA-seq-pe-unstranded-with-sjdb.sh
#!/bin/bash
#SBATCH --job-name=qc
#SBATCH --output=${PROJECT_DIR}/processing/rna_seq/logs/qc_gen.RNA-seq-pe-unstranded-with-sjdb.out

source ${CONDA_ACTIVATE} ${QC_ENV}
cd ${PROJECT_DIR}/processing/rna_seq/RNA-seq-pe-unstranded-with-sjdb

python ${SOFTWARE_DIR}/cwl/bin/generate_stats_rnaseq_paired_end.py ./ \
    -samples $(/bin/ls -1 *PBC.txt | sed 's@.PBC.txt@@') \
> qc.txt

Generate QCs for RNA-seq-pe-unstranded-with-sjdb

In [ ]:
%%script --out blocking_job_str bash -s "$blocking_job"
sbatch -p "$SLURM_PARTITION" \
 --depend afterok:$1 \
 "$PROJECT_DIR/processing/rna_seq/scripts/generate_qc_cell_RNA-seq-pe-unstranded-with-sjdb.sh"

Extract blocking job id

In [ ]:
import re
blocking_job = re.match('Submitted batch job (\d+).*', blocking_job_str).group(1)